# Physics-Informed Graph Neural Network for Distribution Grid State Estimation Under High Renewable Penetration

Companion notebook for the ICDLAI-2026 paper. It reproduces every experiment and figure in the paper:

1. Builds the IEEE 33-bus feeder, 30 radial switching layouts and random load and behind-the-meter PV scenarios (0–150% penetration).
2. Simulates noisy, partly missing measurements and pseudo-measurements.
3. Trains the MLP, GNN, PI-MLP and PI-GNN estimators.
4. Evaluates classical WLS and every network with and without physics-consistent inference (PCI).
5. Prints Tables 2–4 and draws Figures 1–4.

**Runtime.** `MODE = "quick"` runs in about 5–15 minutes on a laptop CPU. It uses less data, fewer epochs and one seed, so its numbers are less accurate than the paper's but show the same main trends. `MODE = "paper"` uses the paper's settings (16,000 training snapshots, 30 epochs, 3 seeds) and takes several hours on CPU; a GPU is not required.

Works in Jupyter or Google Colab. Python 3.9+.

In [ ]:
%pip install -q torch pandapower numpy scipy pandas matplotlib

## 1. Settings

In [ ]:
MODE = "quick"          # "quick" or "paper"

if MODE == "paper":
    N_TRAIN, N_VAL, EPOCHS, SEEDS = 16000, 1500, 30, [0, 1, 2]
    N_MAIN, N_SWEEP, N_PEN, N_STRESS = 2000, 1000, 700, 1000
    LAMBDA_ABLATION = [1e-4, 1e-2]      # extra PI-GNN runs for Table 4 (seed 0)
else:
    N_TRAIN, N_VAL, EPOCHS, SEEDS = 4000, 500, 12, [0]
    N_MAIN, N_SWEEP, N_PEN, N_STRESS = 500, 300, 200, 300
    LAMBDA_ABLATION = []
LAMBDA = 1e-3                           # physics weight selected on validation in the paper
THREADS = 2

import torch; torch.set_num_threads(THREADS)

## 2. Feeder model, AC power flow, PV scenarios and measurement model
The cell writes `grid.py` so the other modules can import it.

In [ ]:
%%writefile grid.py
"""IEEE 33-bus feeder, reconfigurations, AC power flow, PV scenarios, measurement model."""
import numpy as np
import pandapower.networks as pn

VBASE_KV, SBASE = 12.66, 1.0
ZBASE = VBASE_KV ** 2 / SBASE
NB = 33

_net = pn.case33bw()
LINES = _net.line[["from_bus", "to_bus"]].values.astype(int)          # 37 x 2
ZL = (_net.line.r_ohm_per_km.values + 1j * _net.line.x_ohm_per_km.values) * _net.line.length_km.values / ZBASE
BASE_IN = _net.line.in_service.values.astype(bool)                     # 32 sectionalizing in service
LOAD_P = np.zeros(NB); LOAD_Q = np.zeros(NB)
LOAD_P[_net.load.bus.values] = _net.load.p_mw.values / SBASE
LOAD_Q[_net.load.bus.values] = _net.load.q_mvar.values / SBASE
PEAK = LOAD_P.sum()

# ---------- measurement placement ----------
VM_BUSES = np.array([5, 11, 17, 21, 24, 29, 32])     # voltage-magnitude meters (field)
PMU_BUSES = np.array([17, 32])                         # micro-PMU angle measurements
FLOW_LINES = np.array([1, 17, 21, 24])                 # line indices with P/Q flow meters (field)
SIG_VM, SIG_V0, SIG_TH = 0.003, 0.002, 2e-4            # p.u., p.u., rad
SIG_FLOW_REL, SIG_FLOW_ABS = 0.02, 2e-3
SIG_S0_REL = 0.01
SIG_LOAD_REL = 0.20                                    # pseudo-measurement error of load forecasts
SIG_PV_SITE = 0.20                                     # site-level irradiance deviation from feeder sensor


def is_radial_connected(mask):
    if mask.sum() != NB - 1:
        return False
    parent = list(range(NB))
    def f(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]; a = parent[a]
        return a
    for (u, v) in LINES[mask]:
        ru, rv = f(u), f(v)
        if ru == rv:
            return False
        parent[ru] = rv
    return True


def make_topologies(n, rng):
    """Base radial configuration plus n-1 distinct radial reconfigurations (1-3 branch exchanges)."""
    topos = [BASE_IN.copy()]
    seen = {BASE_IN.tobytes()}
    ties = np.where(~BASE_IN)[0]
    while len(topos) < n:
        m = BASE_IN.copy()
        for _ in range(rng.integers(1, 4)):
            t = rng.choice(ties[~m[ties]]) if (~m[ties]).any() else None
            if t is None:
                break
            m[t] = True
            cand = [l for l in np.where(m)[0] if l != 0 and l != t]
            rng.shuffle(cand)
            for l in cand:
                m[l] = False
                if is_radial_connected(m):
                    break
                m[l] = True
        if is_radial_connected(m) and m.tobytes() not in seen:
            # reject configurations with extreme voltage drop at nominal peak load
            V = power_flow(np.array([build_ybus(m)]), np.zeros(1, int), -(LOAD_P + 1j * LOAD_Q)[None])
            if V is not None and np.abs(V[0]).min() > 0.88:
                seen.add(m.tobytes()); topos.append(m)
    return np.array(topos)


def build_ybus(mask):
    Y = np.zeros((NB, NB), complex)
    for (u, v), z, on in zip(LINES, ZL, mask):
        if on:
            y = 1 / z
            Y[u, u] += y; Y[v, v] += y; Y[u, v] -= y; Y[v, u] -= y
    return Y


def power_flow(Ybus_stack, topo_idx, S, tol=1e-10, iters=200):
    """Batched Z-bus fixed-point AC power flow. S: complex net injections (gen - load), slack bus 0 at 1.0 pu."""
    B = S.shape[0]
    V = np.ones((B, NB), complex)
    Zll = np.stack([np.linalg.inv(Y[1:, 1:]) for Y in Ybus_stack])
    Yl0 = np.stack([Y[1:, 0] for Y in Ybus_stack])
    Zt, Yt = Zll[topo_idx], Yl0[topo_idx]
    for it in range(iters):
        I = np.conj(S[:, 1:] / V[:, 1:]) - Yt * V[:, :1]
        Vn = np.einsum("bij,bj->bi", Zt, I)
        err = np.abs(Vn - V[:, 1:]).max()
        V[:, 1:] = Vn
        if err < tol:
            return V
    return V if err < 1e-6 else None


def generate(n, topo_ids, Ybus_stack, rng, pen_range=(0.0, 1.0), pen_fixed=None):
    """Sample operating points. Returns dict with true states and scenario variables."""
    topo = rng.choice(topo_ids, n)
    pen = np.full(n, pen_fixed) if pen_fixed is not None else rng.uniform(*pen_range, n)
    lam = rng.uniform(0.3, 1.1, n)[:, None] * np.clip(1 + 0.15 * rng.standard_normal((n, NB)), 0.4, 1.6)
    pf_scale = np.clip(1 + 0.1 * rng.standard_normal((n, NB)), 0.7, 1.3)
    Pl, Ql = LOAD_P * lam, LOAD_Q * lam * pf_scale
    # PV sites: random 30-60% of load buses, capacity share random, total = pen * peak load
    cap = np.zeros((n, NB))
    for i in range(n):
        k = rng.integers(10, 20)
        sites = rng.choice(np.arange(1, NB), k, replace=False)
        w = rng.gamma(2.0, 1.0, k)
        cap[i, sites] = pen[i] * PEAK * w / w.sum()
    g = rng.beta(2.0, 1.5, n)                                      # feeder clear-sky/irradiance index
    site = np.clip(g[:, None] * (1 + SIG_PV_SITE * rng.standard_normal((n, NB))), 0, 1.0)
    Ppv = cap * site
    S = (Ppv - Pl) - 1j * Ql
    S[:, 0] = 0
    V = power_flow(Ybus_stack, topo, S)
    Ytopo = Ybus_stack[topo]
    Sinj = V * np.conj(np.einsum("bij,bj->bi", Ytopo, V))          # includes slack injection
    return dict(topo=topo, pen=pen, V=V, Sinj=Sinj, cap=cap, g=g, Pl=Pl, Ql=Ql, Ppv=Ppv)


def line_flows(V, topo_masks, lines_idx):
    """Complex power flow from->to on given lines (in-service lines only; zero otherwise)."""
    f, t = LINES[lines_idx, 0], LINES[lines_idx, 1]
    y = 1 / ZL[lines_idx]
    Sft = V[:, f] * np.conj((V[:, f] - V[:, t]) * y)
    return Sft * topo_masks[:, lines_idx]


def measure(d, topos, rng, noise_scale=1.0, drop=0.0):
    """Noisy, partially missing measurements + pseudo-measurements (same inputs for every estimator)."""
    n = len(d["topo"])
    V, Sinj = d["V"], d["Sinj"]
    ns = noise_scale
    z = {}
    z["v0"] = np.abs(V[:, 0]) + ns * SIG_V0 * rng.standard_normal(n)
    s0 = Sinj[:, 0]
    z["p0"] = s0.real + ns * SIG_S0_REL * np.abs(s0.real) * rng.standard_normal(n) + ns * 1e-3 * rng.standard_normal(n)
    z["q0"] = s0.imag + ns * SIG_S0_REL * np.abs(s0.imag) * rng.standard_normal(n) + ns * 1e-3 * rng.standard_normal(n)
    z["vm"] = np.abs(V[:, VM_BUSES]) + ns * SIG_VM * rng.standard_normal((n, len(VM_BUSES)))
    z["th"] = np.angle(V[:, PMU_BUSES]) + ns * SIG_TH * rng.standard_normal((n, len(PMU_BUSES)))
    tm = topos[d["topo"]]
    F = line_flows(V, tm, FLOW_LINES)
    sf = lambda x: ns * (SIG_FLOW_REL * np.abs(x) + SIG_FLOW_ABS)
    z["pf"] = F.real + sf(F.real) * rng.standard_normal(F.shape)
    z["qf"] = F.imag + sf(F.imag) * rng.standard_normal(F.shape)
    # availability masks for field measurements (substation always available); lines out of service have no meter
    dr = np.asarray(drop, float).reshape(-1, 1) if np.ndim(drop) else drop
    z["m_vm"] = (rng.random((n, len(VM_BUSES))) >= dr).astype(float)
    z["m_th"] = (rng.random((n, len(PMU_BUSES))) >= dr).astype(float)
    z["m_f"] = (rng.random((n, len(FLOW_LINES))) >= dr).astype(float) * tm[:, FLOW_LINES]
    # pseudo-measurements: load forecast (20% error) and PV estimate = capacity x feeder irradiance sensor
    pl = d["Pl"] * (1 + SIG_LOAD_REL * rng.standard_normal(d["Pl"].shape))
    ql = d["Ql"] * (1 + SIG_LOAD_REL * rng.standard_normal(d["Ql"].shape))
    g_meas = np.clip(d["g"] + 0.02 * rng.standard_normal(n), 0, 1)
    ppv = d["cap"] * g_meas[:, None]
    z["pp"] = ppv - pl
    z["qp"] = -ql
    z["sig_pp"] = np.sqrt((SIG_LOAD_REL * pl) ** 2 + (SIG_PV_SITE * ppv) ** 2 + 1e-8)
    z["sig_qp"] = np.sqrt((SIG_LOAD_REL * ql) ** 2 + 1e-8)
    z["cap"] = d["cap"]; z["g"] = g_meas
    for k in ["pp", "qp", "sig_pp", "sig_qp"]:
        z[k][:, 0] = 0
    return z


## 3. Measurement function, WLS / physics-consistent inference, and the neural estimators
`h_and_jac` gives the AC measurement function and its analytic Jacobian; `wls_fast` is the batched Gauss–Newton solver used both for classical WLS and, with a prior, for PCI (Eq. 5 in the paper). `GNNEstimator` is the edge-conditioned message-passing network (Section 4.2).

In [ ]:
%%writefile models.py
"""Measurement model h(x) in torch, WLS estimator, MLP / GNN / PI-GNN estimators."""
import numpy as np
import torch
import torch.nn as nn
import grid as G

NB = G.NB
VMB = torch.tensor(G.VM_BUSES); PMB = torch.tensor(G.PMU_BUSES)
FL = torch.tensor(G.FLOW_LINES)
LF = torch.tensor(G.LINES[G.FLOW_LINES, 0]); LT = torch.tensor(G.LINES[G.FLOW_LINES, 1])
YF = 1 / G.ZL[G.FLOW_LINES]


class Batch:
    """Torch view of a set of samples: measurements, weights, topology admittances, targets."""
    def __init__(self, d, z, topos, Ystack, dtype=torch.float32):
        t = lambda a: torch.as_tensor(np.asarray(a), dtype=dtype)
        n = len(d["topo"])
        self.n = n
        self.topo = torch.as_tensor(d["topo"])
        self.Gs, self.Bs = t(Ystack.real), t(Ystack.imag)
        self.tmask = t(topos.astype(float))
        # measurement vector z and weights w = mask / sigma (nominal sigma, as assumed by the operator)
        zs, ws = [], []
        def add(val, sig, mask=None):
            val = t(val); sig = t(sig)
            if val.dim() == 1: val = val[:, None]
            if sig.dim() == 1: sig = sig[:, None]
            sig = sig.expand_as(val)
            m = torch.ones_like(val) if mask is None else t(mask)
            zs.append(val); ws.append(m / sig)
        add(z["v0"], torch.tensor(G.SIG_V0))
        add(z["p0"], G.SIG_S0_REL * np.abs(z["p0"]) + 1e-3)
        add(z["q0"], G.SIG_S0_REL * np.abs(z["q0"]) + 1e-3)
        add(z["vm"], torch.full((n, len(G.VM_BUSES)), G.SIG_VM), z["m_vm"])
        add(z["th"], torch.full((n, len(G.PMU_BUSES)), G.SIG_TH), z["m_th"])
        add(z["pf"], G.SIG_FLOW_REL * np.abs(z["pf"]) + G.SIG_FLOW_ABS, z["m_f"])
        add(z["qf"], G.SIG_FLOW_REL * np.abs(z["qf"]) + G.SIG_FLOW_ABS, z["m_f"])
        add(z["pp"][:, 1:], z["sig_pp"][:, 1:])
        add(z["qp"][:, 1:], z["sig_qp"][:, 1:])
        self.z = torch.cat(zs, 1) * 1.0
        self.w = torch.cat(ws, 1)
        self.z = torch.where(self.w > 0, self.z, torch.zeros_like(self.z))
        self.Vm = t(np.abs(d["V"])); self.th = t(np.angle(d["V"]))
        self.raw = z
        self.dtype = dtype
        self.idx = torch.arange(n)
        self.nf = node_features(self)
        self.ei, self.ef = edge_features(self)
        fm = np.zeros((n, 12), np.float32)
        fm[:, 0:4] = z["pf"] * 5 * z["m_f"]; fm[:, 4:8] = z["qf"] * 5 * z["m_f"]; fm[:, 8:12] = z["m_f"]
        self.fm = torch.from_numpy(fm)

    def sub(self, idx):
        b = object.__new__(Batch)
        for k, v in self.__dict__.items():
            if k in ("Gs", "Bs", "tmask", "dtype", "raw", "n"):
                setattr(b, k, v)
            else:
                setattr(b, k, v[idx])
        b.n = len(idx)
        return b


def h_meas(Vm, th, Gm, Bm, tmask_row):
    """Measurement function for a batch. Vm, th: [B, NB]; Gm, Bm: [B, NB, NB]; tmask_row: [B, 37]."""
    Vr, Vi = Vm * torch.cos(th), Vm * torch.sin(th)
    Ir = torch.einsum("bij,bj->bi", Gm, Vr) - torch.einsum("bij,bj->bi", Bm, Vi)
    Ii = torch.einsum("bij,bj->bi", Gm, Vi) + torch.einsum("bij,bj->bi", Bm, Vr)
    P = Vr * Ir + Vi * Ii
    Q = Vi * Ir - Vr * Ii
    yr = torch.as_tensor(YF.real, dtype=Vm.dtype); yi = torch.as_tensor(YF.imag, dtype=Vm.dtype)
    dr, di = Vr[:, LF] - Vr[:, LT], Vi[:, LF] - Vi[:, LT]
    Ifr, Ifi = dr * yr - di * yi, dr * yi + di * yr
    Pf = (Vr[:, LF] * Ifr + Vi[:, LF] * Ifi) * tmask_row[:, FL]
    Qf = (Vi[:, LF] * Ifr - Vr[:, LF] * Ifi) * tmask_row[:, FL]
    return torch.cat([Vm[:, :1], P[:, :1], Q[:, :1], Vm[:, VMB], th[:, PMB], Pf, Qf, P[:, 1:], Q[:, 1:]], 1)


def residual(b, Vm, th):
    h = h_meas(Vm, th, b.Gs[b.topo], b.Bs[b.topo], b.tmask[b.topo])
    return (b.z - h) * b.w


# ---------------------------------------------------------------- learned estimators
VSCALE, TSCALE = 20.0, 30.0          # standardisation of outputs: (Vm-1)*20, th*30


def node_features(b):
    z = b.raw
    n = b.n
    idx = b.idx.numpy()
    f = np.zeros((n, NB, 11), np.float32)
    f[:, 0, 0] = (z["v0"][idx] - 1) * 20; f[:, 0, 1] = 1
    f[:, G.VM_BUSES, 0] = (z["vm"][idx] - 1) * 20 * z["m_vm"][idx]; f[:, G.VM_BUSES, 1] = z["m_vm"][idx]
    f[:, 0, 3] = 1                                                   # slack angle reference known
    f[:, G.PMU_BUSES, 2] = z["th"][idx] * 30 * z["m_th"][idx]; f[:, G.PMU_BUSES, 3] = z["m_th"][idx]
    f[:, :, 4] = z["pp"][idx] * 5; f[:, :, 5] = z["qp"][idx] * 5
    f[:, :, 6] = z["cap"][idx] * 5
    f[:, :, 7] = z["g"][idx][:, None]
    f[:, 0, 8] = 1
    f[:, 0, 9] = z["p0"][idx] / 2; f[:, 0, 10] = z["q0"][idx] / 2
    return torch.from_numpy(f)


def edge_features(b):
    """Directed edges of in-service lines (32 per sample -> 64 directed), features [r, x, dir, Pf, Qf, mask]."""
    z = b.raw
    idx = b.idx.numpy()
    tm = b.tmask[b.topo].numpy().astype(bool)
    lines = np.stack([np.where(r)[0] for r in tm])                  # [n, 32]
    src, dst = G.LINES[lines, 0], G.LINES[lines, 1]
    ei = np.concatenate([np.stack([src, dst], -1), np.stack([dst, src], -1)], 1)  # [n, 64, 2]
    zl = G.ZL[lines] * 100
    fm = np.zeros((b.n, 37, 3), np.float32)
    fm[:, G.FLOW_LINES, 0] = z["pf"][idx] * 5 * z["m_f"][idx]
    fm[:, G.FLOW_LINES, 1] = z["qf"][idx] * 5 * z["m_f"][idx]
    fm[:, G.FLOW_LINES, 2] = z["m_f"][idx]
    fl = np.take_along_axis(fm, lines[..., None], 1)
    fwd = np.concatenate([zl.real[..., None], zl.imag[..., None], np.ones_like(zl.real)[..., None], fl], -1)
    bwd = fwd.copy(); bwd[..., 2] = -1; bwd[..., 3:5] *= -1
    ef = np.concatenate([fwd, bwd], 1).astype(np.float32)
    return torch.from_numpy(ei).long(), torch.from_numpy(ef)


def mlp(i, h, o, nl=2):
    layers, d = [], i
    for _ in range(nl):
        layers += [nn.Linear(d, h), nn.SiLU()]; d = h
    return nn.Sequential(*layers, nn.Linear(d, o))


class MLPEstimator(nn.Module):
    def __init__(self, hid=256):
        super().__init__()
        self.net = nn.Sequential(mlp(NB * 11 + 37 + 12, hid, 2 * NB, nl=3))

    def forward(self, b):
        x = torch.cat([b.nf.flatten(1), b.tmask[b.topo], b.fm], 1)
        out = self.net(x)
        return out[:, :NB], out[:, NB:]


class GNNEstimator(nn.Module):
    """Edge-conditioned residual message-passing network on the feeder graph.
    Message m_st = W2 SiLU(A h_s + B h_t + C e_st); update h_t <- LN(h_t + U([h_t, sum_s m_st]))."""
    def __init__(self, hid=64, layers=12):
        super().__init__()
        self.enc = mlp(11, hid, hid, 1)
        self.eenc = mlp(6, hid, hid, 1)
        self.A = nn.ModuleList([nn.Linear(hid, hid) for _ in range(layers)])
        self.Bm = nn.ModuleList([nn.Linear(hid, hid, bias=False) for _ in range(layers)])
        self.C = nn.ModuleList([nn.Linear(hid, hid, bias=False) for _ in range(layers)])
        self.W2 = nn.ModuleList([nn.Linear(hid, hid) for _ in range(layers)])
        self.upd = nn.ModuleList([mlp(2 * hid, hid, hid, 1) for _ in range(layers)])
        self.norm = nn.ModuleList([nn.LayerNorm(hid) for _ in range(layers)])
        self.dec = mlp(hid, hid, 2, 1)

    def forward(self, b):
        nf, ei, ef = b.nf, b.ei, b.ef
        B = b.n
        h = self.enc(nf).reshape(B * NB, -1)
        off = (torch.arange(B) * NB)[:, None]
        s = (ei[..., 0] + off).reshape(-1); t = (ei[..., 1] + off).reshape(-1)
        e = self.eenc(ef).reshape(B * 64, -1)
        for A, Bm, C, W2, upd, nm in zip(self.A, self.Bm, self.C, self.W2, self.upd, self.norm):
            m = W2(nn.functional.silu(A(h)[s] + Bm(h)[t] + C(e)))
            agg = torch.zeros_like(h).index_add_(0, t, m)
            h = nm(h + upd(torch.cat([h, agg], -1)))
        out = self.dec(h).reshape(B, NB, 2)
        return out[..., 0], out[..., 1]


def to_state(vo, to):
    Vm = 1 + vo / VSCALE
    th = to / TSCALE
    th = th - th[:, :1]                                              # slack angle reference
    return Vm, th


def prior_sigma(Vm, th, b, inflate=1.0):
    """Per-state prior std from validation errors of a learned estimator (x = [th_1..th_32, Vm_0..Vm_32])."""
    sv = (Vm - b.Vm).pow(2).mean(0).sqrt()
    st = (th - b.th).pow(2).mean(0).sqrt()[1:]
    return torch.cat([st, sv]).clamp(min=1e-5) * inflate


def map_refine(b, Vm, th, sigma, iters=10):
    """Physics-consistent inference: MAP estimate combining AC measurement model (likelihood) with the
    learned estimate as a Gaussian prior; Gauss-Newton warm-started at the network output."""
    x0 = torch.cat([th[:, 1:], Vm], 1)
    return wls_fast(b, iters=iters, x0=x0, prior=(x0.double(), sigma[None]))


def h_and_jac(x, Yc, tm):
    """Analytic measurement function and Jacobian (complex128). x: [B, 65]; Yc: [B, N, N] complex; tm: [B, 37]."""
    B = x.shape[0]
    th = torch.cat([x.new_zeros(B, 1), x[:, :NB - 1]], 1)
    Vm = x[:, NB - 1:]
    V = torch.polar(Vm, th)
    I = torch.einsum("bij,bj->bi", Yc, V)
    S = V * I.conj()
    Vn = V / Vm
    dVa = 1j * V[:, :, None] * (torch.diag_embed(I) - Yc * V[:, None, :]).conj()
    dVm = V[:, :, None] * (Yc * Vn[:, None, :]).conj() + torch.diag_embed(I.conj() * Vn)
    nl = len(G.FLOW_LINES)
    f, t = LF, LT
    y = torch.as_tensor(YF, dtype=torch.complex128)
    Vf, Vt = V[:, f], V[:, t]
    Sf = Vf * (y * (Vf - Vt)).conj() * tm[:, FL]
    M = 3 + len(G.VM_BUSES) + len(G.PMU_BUSES) + 2 * nl + 2 * (NB - 1)
    J = x.new_zeros(B, M, 2 * NB - 1)
    h = torch.cat([Vm[:, :1], S.real[:, :1], S.imag[:, :1], Vm[:, VMB], th[:, PMB], Sf.real, Sf.imag,
                   S.real[:, 1:], S.imag[:, 1:]], 1)
    J[:, 0, NB - 1] = 1
    J[:, 1, :NB - 1] = dVa[:, 0, 1:].real; J[:, 1, NB - 1:] = dVm[:, 0].real
    J[:, 2, :NB - 1] = dVa[:, 0, 1:].imag; J[:, 2, NB - 1:] = dVm[:, 0].imag
    r = 3
    for k, bb in enumerate(G.VM_BUSES):
        J[:, r + k, NB - 1 + bb] = 1
    r += len(G.VM_BUSES)
    for k, bb in enumerate(G.PMU_BUSES):
        J[:, r + k, bb - 1] = 1
    r += len(G.PMU_BUSES)
    yc = y.conj()
    dSf_dthf = -1j * Vf * yc * Vt.conj()
    dSf_dtht = 1j * Vf * yc * Vt.conj()
    dSf_dVf = (Vf / Vm[:, f]) * yc * (2 * Vf.conj() - Vt.conj())
    dSf_dVt = -Vf * yc * Vt.conj() / Vm[:, t]
    for k in range(nl):
        fk, tk = int(f[k]), int(t[k]); on = tm[:, FL[k]]
        for part, row in ((lambda z: z.real, r + k), (lambda z: z.imag, r + nl + k)):
            if fk > 0: J[:, row, fk - 1] += part(dSf_dthf[:, k]) * on
            if tk > 0: J[:, row, tk - 1] += part(dSf_dtht[:, k]) * on
            J[:, row, NB - 1 + fk] += part(dSf_dVf[:, k]) * on
            J[:, row, NB - 1 + tk] += part(dSf_dVt[:, k]) * on
    r += 2 * nl
    J[:, r:r + NB - 1, :NB - 1] = dVa[:, 1:, 1:].real; J[:, r:r + NB - 1, NB - 1:] = dVm[:, 1:].real
    r += NB - 1
    J[:, r:r + NB - 1, :NB - 1] = dVa[:, 1:, 1:].imag; J[:, r:r + NB - 1, NB - 1:] = dVm[:, 1:].imag
    return h, J


def wls_fast(b, iters=30, tol=1e-7, x0=None, prior=None, chunk=2000):
    outs = []
    for i in range(0, b.n, chunk):
        idx = torch.arange(i, min(b.n, i + chunk))
        s = b.sub(idx)
        x0c = None if x0 is None else x0[idx]
        pc = None if prior is None else (prior[0][idx], prior[1])
        outs.append(_wls_fast(s, iters, tol, x0c, pc))
    return (torch.cat([o[0] for o in outs]), torch.cat([o[1] for o in outs]),
            torch.cat([o[2] for o in outs]), max(o[3] for o in outs))


def _wls_fast(b, iters, tol, x0, prior):
    B = b.n
    x = torch.cat([torch.zeros(B, NB - 1), torch.ones(B, NB)], 1).double() if x0 is None else x0.clone().double()
    Yc = torch.complex(b.Gs[b.topo].double(), b.Bs[b.topo].double())
    tm = b.tmask[b.topo].double()
    z, w2 = b.z.double(), b.w.double() ** 2
    conv = torch.zeros(B, dtype=torch.bool)
    for it in range(iters):
        h, J = h_and_jac(x, Yc, tm)
        JW = J.transpose(1, 2) * w2[:, None, :]
        A = JW @ J
        g = (JW @ (z - h)[..., None])[..., 0]
        if prior is not None:
            xp, sp = prior
            ip = (1 / sp.double() ** 2).expand_as(x)
            A = A + torch.diag_embed(ip)
            g = g - ip * (x - xp.double())
        dx = torch.linalg.solve(A + 1e-10 * torch.eye(A.shape[-1], dtype=A.dtype), g[..., None])[..., 0]
        dx = torch.nan_to_num(dx, nan=0.0)
        x = torch.where(conv[:, None], x, x + dx)
        conv = conv | (dx.abs().max(1).values < tol)
        if conv.all():
            break
    th = torch.cat([torch.zeros(B, 1, dtype=x.dtype), x[:, :NB - 1]], 1)
    return x[:, NB - 1:].float(), th.float(), conv, it + 1


## 4. Training loop (supervised loss + optional physics loss, Eq. 4) and metrics

In [ ]:
%%writefile train.py
import numpy as np, torch, time, math
import grid as G, models as M



def train_model(kind, d_tr, d_va, T, Ys, seed, lam=0.0, label_frac=1.0, epochs=40, bs=128,
                lr=2e-3, resample_every=4, log=None, verbose=False):
    """kind in {'mlp','gnn'}; lam>0 adds the physics (measurement-consistency) loss -> PI-GNN / PI-MLP.
    label_frac<1: only that fraction of training states has ground-truth labels (physics loss uses all)."""
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    model = M.MLPEstimator() if kind == "mlp" else M.GNNEstimator()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    n = len(d_tr["topo"])
    lab = torch.zeros(n, dtype=torch.bool)
    lab[torch.from_numpy(rng.permutation(n)[: max(1, int(round(label_frac * n)))])] = True
    steps = epochs * math.ceil(n / bs)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps, pct_start=0.1)
    zva = G.measure(d_va, T, np.random.default_rng(999), drop=np.random.default_rng(998).uniform(0, 0.6, len(d_va["topo"])))
    bva = M.Batch(d_va, zva, T, Ys)
    best, best_state = 1e9, None
    for ep in range(epochs):
        if ep % resample_every == 0:        # fresh noise + random meter outages (augmentation)
            z = G.measure(d_tr, T, rng, drop=rng.uniform(0, 0.6, n))
            btr = M.Batch(d_tr, z, T, Ys)
        model.train()
        perm = torch.randperm(n)
        for i in range(0, n, bs):
            idx = perm[i:i + bs]
            b = btr.sub(idx)
            vo, to = model(b)
            Vm, th = M.to_state(vo, to)
            b_lab = lab[idx]
            loss = torch.zeros(())
            if b_lab.any():
                sup = ((Vm - b.Vm) * M.VSCALE) ** 2 + ((th - b.th) * M.TSCALE) ** 2
                loss = loss + sup[b_lab].mean()
            if lam > 0:
                ramp = min(1.0, ep / max(1, int(0.3 * epochs)))        # curriculum: physics term ramps in
                r = M.residual(b, Vm, th)
                loss = loss + ramp * lam * torch.nn.functional.huber_loss(r, torch.zeros_like(r), delta=3.0, reduction="mean")
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step(); sched.step()
        model.eval()
        with torch.no_grad():
            Vm, th = predict(model, bva)
            val = ((Vm - bva.Vm).abs().mean() * M.VSCALE + (th - bva.th).abs().mean() * M.TSCALE).item()
        if val < best:
            best = val; best_state = {k: v.clone() for k, v in model.state_dict().items()}
        if verbose:
            print(f"{kind} lam={lam} ep{ep} loss={loss.item():.4f} val={val:.4f}", flush=True)
    model.load_state_dict(best_state)
    return model


@torch.no_grad()
def predict(model, b, bs=1024):
    model.eval()
    outs = []
    for i in range(0, b.n, bs):
        s = b.sub(torch.arange(i, min(b.n, i + bs)))
        outs.append(M.to_state(*model(s)))
    return torch.cat([o[0] for o in outs]), torch.cat([o[1] for o in outs])


def metrics(Vm, th, b, conv=None):
    ev = (Vm - b.Vm).abs(); et = (th - b.th).abs()
    r = M.residual(b, Vm, th)
    rt = r[:, :3 + len(G.VM_BUSES) + len(G.PMU_BUSES) + 2 * len(G.FLOW_LINES)]
    wt = b.w[:, :rt.shape[1]] > 0
    out = dict(
        mae_v=ev.mean().item() * 1e3, rmse_v=ev.pow(2).mean().sqrt().item() * 1e3,
        mae_t=et.mean().item() * 1e3, rmse_t=et.pow(2).mean().sqrt().item() * 1e3,
        p99_maxv=torch.quantile(ev.max(1).values, 0.99).item() * 1e3,
        viol=(ev.max(1).values > 0.01).float().mean().item() * 100,   # % snapshots with any |dV| > 0.01 pu
        chi_rt=((rt ** 2).sum(1) / wt.sum(1).clamp(min=1)).mean().item(),  # normalised real-time residual
    )
    if conv is not None:
        out["fail"] = (1 - conv.float().mean().item()) * 100
    return out


## 5. Generate layouts, training data and all test suites

In [ ]:
import numpy as np, torch, time, importlib
import grid as G, models as M, train as TR
for m in (G, M, TR): importlib.reload(m)

rng = np.random.default_rng(2026)
T = G.make_topologies(30, rng)                       # 30 radial layouts
Ys = np.array([G.build_ybus(m) for m in T])
seen, unseen = np.arange(15), np.arange(15, 30)      # 15 for training/testing, 15 never seen in training

d_tr = G.generate(N_TRAIN, seen, Ys, rng)            # PV penetration uniform in [0, 100%]
d_va = G.generate(N_VAL, seen, Ys, rng)
suites = {}
main = G.generate(N_MAIN, seen, Ys, rng); suites["main"] = (main, G.measure(main, T, rng))
un = G.generate(N_MAIN, unseen, Ys, rng); suites["unseen"] = (un, G.measure(un, T, rng))
sweep = G.generate(N_SWEEP, seen, Ys, rng)
for dr in [0.0, 0.25, 0.5, 0.75, 1.0]:
    suites[f"drop{dr}"] = (sweep, G.measure(sweep, T, rng, drop=dr))
for ns in [0.5, 1.0, 2.0, 4.0]:
    suites[f"noise{ns}"] = (sweep, G.measure(sweep, T, rng, noise_scale=ns))
for pen in [0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5]:
    d = G.generate(N_PEN, seen, Ys, rng, pen_fixed=pen); suites[f"pen{pen}"] = (d, G.measure(d, T, rng))
d = G.generate(N_STRESS, unseen, Ys, rng, pen_fixed=1.25)
suites["stress"] = (d, G.measure(d, T, rng, drop=0.5, noise_scale=2.0))
va_meas = G.measure(d_va, T, np.random.default_rng(7), drop=np.random.default_rng(8).uniform(0, 0.6, N_VAL))

B = {k: M.Batch(d, z, T, Ys) for k, (d, z) in suites.items()}
b_va = M.Batch(d_va, va_meas, T, Ys)
print(len(T), "layouts;", N_TRAIN, "training snapshots;", len(B), "test suites")

## 6. Train the estimators
`mlp` / `gnn` are purely data-driven; `pimlp` / `pign` add the physics loss with weight λ.

In [ ]:
RUNS = [("mlp", "mlp", 0.0), ("pimlp", "mlp", LAMBDA), ("gnn", "gnn", 0.0), ("pign", "gnn", LAMBDA)]
RUNS += [(f"pign_{lam:g}", "gnn", lam) for lam in LAMBDA_ABLATION]
models = {}
for tag, kind, lam in RUNS:
    seeds = SEEDS if not tag.startswith("pign_") else [0]
    for s in seeds:
        t = time.time()
        models[(tag, s)] = TR.train_model(kind, d_tr, d_va, T, Ys, s, lam=lam, epochs=EPOCHS)
        print(f"{tag:12s} seed {s}: trained in {time.time() - t:.0f} s")

## 7. Evaluate WLS and every model, with and without physics-consistent inference (PCI)

In [ ]:
R = {}                                    # R[suite][method] = list of metric dicts (one per seed)
def put(s, m, v): R.setdefault(s, {}).setdefault(m, []).append(v)

for k, b in B.items():                    # classical WLS from a flat start
    Vm, th, conv, _ = M.wls_fast(b)
    put(k, "WLS", TR.metrics(Vm, th, b, conv))

def pick_inflation(model):                # choose the prior-uncertainty factor on validation data
    Vv, tv = TR.predict(model, b_va); s = M.prior_sigma(Vv, tv, b_va)
    best = None
    for inf in [1.0, 2.0, 4.0, 8.0]:
        Vr, tr, _, _ = M.map_refine(b_va, Vv, tv, s * inf)
        score = ((Vr - b_va.Vm).abs().mean() * M.VSCALE + (tr - b_va.th).abs().mean() * M.TSCALE).item()
        if best is None or score < best[0]: best = (score, inf)
    return s * best[1]

for (tag, s), model in models.items():
    sig = pick_inflation(model)
    for k, b in B.items():
        Vm, th = TR.predict(model, b)
        put(k, tag, TR.metrics(Vm, th, b))
        Vr, tr, c, _ = M.map_refine(b, Vm, th, sig)
        put(k, tag + "+PCI", TR.metrics(Vr, tr, b, c))
    print("evaluated", tag, "seed", s)

## 8. Tables 2–4

In [ ]:
import pandas as pd
NAMES = {"WLS": "WLS", "mlp": "MLP", "pimlp": "PI-MLP", "gnn": "GNN", "pign": "PI-GNN",
         "mlp+PCI": "MLP + PCI", "gnn+PCI": "GNN + PCI", "pign+PCI": "PI-GNN + PCI (proposed)"}
def mean(s, m, key): return float(np.mean([r[key] for r in R[s][m]]))
def table(suite, methods=NAMES):
    rows = [{"Method": NAMES.get(m, m), "V MAE (1e-3 pu)": mean(suite, m, "mae_v"), "Angle MAE (mrad)": mean(suite, m, "mae_t"),
             "Large-error %": mean(suite, m, "viol"), "chi2_rt": mean(suite, m, "chi_rt")} for m in methods if m in R[suite]]
    return pd.DataFrame(rows).round(3)
print("Table 2 - nominal test set"); display(table("main"))
print("Table 3 - unseen layouts"); display(table("unseen"))
print("Table 3 - combined stress"); display(table("stress"))
if LAMBDA_ABLATION:
    print("Table 4 - physics weight")
    display(pd.DataFrame([{"lambda": t, "Nominal": mean("main", t, "mae_v"), "Unseen": mean("unseen", t, "mae_v"),
                           "Nominal + PCI": mean("main", t + "+PCI", "mae_v"), "Unseen + PCI": mean("unseen", t + "+PCI", "mae_v")}
                          for t in ["gnn", "pign"] + [f"pign_{l:g}" for l in LAMBDA_ABLATION]]).round(3))

## 9. Figures 1 and 2 (framework and feeder)

In [ ]:
import matplotlib; import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import numpy as np, grid as G, os
os.makedirs('figs', exist_ok=True)
plt.rcParams.update({"font.family":"DejaVu Sans","font.size":9})
INK="#0b0b0b"; MUTED="#52514e"; GRID="#d9d8d3"
# ---- Fig 2 feeder (0-indexed internally, labels 1-indexed)
pos={}
for i in range(18): pos[i]=(i,0)
for k,b in enumerate(range(18,22)): pos[b]=(1+k+1,-1.6)
for k,b in enumerate(range(22,25)): pos[b]=(2+k+1,1.6)
for k,b in enumerate(range(25,33)): pos[b]=(5+k+1,-3.2)
fig,ax=plt.subplots(figsize=(8.2,3.4))
for li,(u,v) in enumerate(G.LINES):
    x=[pos[u][0],pos[v][0]]; y=[pos[u][1],pos[v][1]]
    if G.BASE_IN[li]:
        lw=2.6 if li in G.FLOW_LINES else 1.4
        ax.plot(x,y,color=INK if li not in G.FLOW_LINES else "#2a78d6",lw=lw,zorder=1)
    else:
        ax.plot(x,y,color=MUTED,lw=1,ls=(0,(3,2)),zorder=0)
for b,(x,y) in pos.items():
    if b==0: ax.scatter(x,y,s=140,marker="s",color=INK,zorder=3)
    elif b in G.PMU_BUSES: ax.scatter(x,y,s=110,marker="D",color="#eb6834",edgecolor="white",lw=1.5,zorder=3)
    elif b in G.VM_BUSES: ax.scatter(x,y,s=90,marker="o",color="#1baf7a",edgecolor="white",lw=1.5,zorder=3)
    else: ax.scatter(x,y,s=26,color="white",edgecolor=INK,lw=1,zorder=3)
    ax.annotate(str(b+1),(x,y),xytext=(0,7),textcoords="offset points",ha="center",fontsize=7,color=MUTED)
from matplotlib.lines import Line2D
h=[Line2D([],[],marker="s",ls="",color=INK,ms=8,label="Substation (|V|, P, Q)"),
   Line2D([],[],marker="o",ls="",color="#1baf7a",ms=8,label="|V| meter"),
   Line2D([],[],marker="D",ls="",color="#eb6834",ms=8,label="|V| meter + µPMU angle"),
   Line2D([],[],color="#2a78d6",lw=2.6,label="Line with P/Q flow meter"),
   Line2D([],[],color=MUTED,lw=1,ls=(0,(3,2)),label="Normally open tie line")]
ax.legend(handles=h,loc="upper right",frameon=False,fontsize=7.5,ncol=1)
ax.set_axis_off(); ax.set_xlim(-0.6,17.6); ax.set_ylim(-3.8,2.4)
plt.tight_layout(); plt.savefig("figs/fig2_feeder.png",dpi=220); plt.show()

# ---- Fig 1 framework
fig,ax=plt.subplots(figsize=(8.4,3.3)); ax.set_axis_off(); ax.set_xlim(0,100); ax.set_ylim(0,40)
def box(x,y,w,h,t,fc="#f3f2ee",ec=INK,bold=False):
    ax.add_patch(FancyBboxPatch((x,y),w,h,boxstyle="round,pad=0.4,rounding_size=1.2",fc=fc,ec=ec,lw=1))
    ax.text(x+w/2,y+h/2,t,ha="center",va="center",fontsize=8,fontweight="bold" if bold else "normal",color=INK)
def arr(x1,y1,x2,y2,ls="-"):
    ax.add_patch(FancyArrowPatch((x1,y1),(x2,y2),arrowstyle="-|>",mutation_scale=10,color=INK,lw=1,ls=ls))
box(1,24,17,12,"Sparse real-time\nmeasurements z, m\n(SCADA, meters, µPMU)")
box(1,6,17,12,"Pseudo-measurements\n(load forecast,\nPV cap. × irradiance)")
box(23,15,21,12,"Feeder graph G(τ)\nnodes: |V|, θ, P̃, Q̃, C, ĝ\nedges: r, x, P/Q flows")
box(49,15,16,12,"Edge-conditioned\nmessage passing\n(12 layers)",fc="#dbe8f8",bold=True)
box(71,24,13,10,"Learned prior\nx̂_NN, Σ_p",fc="#dbe8f8")
box(69,3,30,13,"Physics-consistent inference\n(uses z and Y(τ))\nMAP: ‖W(z−h(x))‖² + ‖x−x̂_NN‖²_Σp⁻¹\nwarm-started Gauss–Newton",fc="#fbe3d7")
box(88,24,11,10,"State\nestimate\n|V|, θ",fc="#e9f5ef",bold=True)
box(38,1,27,9,"Training loss: supervised\n+ λ·AC measurement residual",fc="#fbe3d7")
arr(18.5,30,24,23); arr(18.5,12,24,19); arr(44.5,21,49,21); arr(65.5,23,71,28)
arr(77.5,23.5,80,16.5); arr(92,16.5,93.5,23.5); arr(84.5,29,88,29,ls="--")
arr(55,10.5,55,14.5)
ax.text(86.2,30.6,"fast path",fontsize=6.5,color=MUTED,ha="center")
plt.tight_layout(); plt.savefig("figs/fig1_framework.png",dpi=220); plt.show()


## 10. Figures 3 and 4 (robustness sweeps and physical consistency)

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 8.5, "axes.spines.top": False, "axes.spines.right": False})
def agg(s, m, key):
    v = np.array([r[key] for r in R[s][m]]); return v.mean(), v.std()
C = {"mlp": "#eda100", "gnn": "#2a78d6", "pign": "#eb6834"}; LAB = {"mlp": "MLP", "gnn": "GNN", "pign": "PI-GNN"}
MK = {"mlp": "s", "gnn": "o", "pign": "D"}
sweeps = [("drop", [0.0, 0.25, 0.5, 0.75, 1.0], "Field-meter outage rate (%)", 100),
          ("noise", [0.5, 1.0, 2.0, 4.0], "Noise scale (x nominal)", 1),
          ("pen", [0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5], "PV penetration (% of peak load)", 100)]
fig, axs = plt.subplots(2, 3, figsize=(9.2, 5.4))
for row, suf in enumerate(["", "+PCI"]):
    for col, (pre, lv, xl, sc) in enumerate(sweeps):
        ax = axs[row, col]; X = [l * sc for l in lv]
        ax.plot(X, [agg(f"{pre}{l}", "WLS", "mae_v")[0] for l in lv], color="#52514e", lw=2, ls=(0, (4, 2)), label="WLS")
        for m in ["mlp", "gnn", "pign"]:
            a = np.array([agg(f"{pre}{l}", m + suf, "mae_v") for l in lv])
            ax.plot(X, a[:, 0], color=C[m], lw=2, marker=MK[m], ms=5, mec="white", label=LAB[m] + suf.replace("+", " + "))
            ax.fill_between(X, a[:, 0] - a[:, 1], a[:, 0] + a[:, 1], color=C[m], alpha=0.18, lw=0)
        if pre == "pen": ax.axvspan(100, 150, color="#e9e8e3", zorder=0)
        ax.set_xlabel(xl); ax.grid(axis="y", color="#e9e8e3")
        if col == 0: ax.set_ylabel("|V| MAE (1e-3 p.u.)"); ax.legend(frameon=False, fontsize=7.5)
    axs[row, 0].set_title("(a) networks alone" if row == 0 else "(b) with physics-consistent inference", loc="left", fontweight="bold")
plt.tight_layout(); plt.savefig("figs/fig3_sweeps.png", dpi=220); plt.show()

meths = ["WLS", "mlp", "pimlp", "gnn", "pign", "mlp+PCI", "gnn+PCI", "pign+PCI"]
names = ["WLS", "MLP", "PI-MLP", "GNN", "PI-GNN", "MLP+PCI", "GNN+PCI", "PI-GNN+PCI"]
fig, ax = plt.subplots(figsize=(9.2, 3.2)); x = np.arange(len(meths))
for k, (s, lab, c) in enumerate([("main", "Nominal", "#2a78d6"), ("unseen", "Unseen layouts", "#1baf7a"), ("stress", "Combined stress", "#eb6834")]):
    ax.bar(x + (k - 1) * 0.26, [agg(s, m, "chi_rt")[0] for m in meths], 0.23, color=c, label=lab)
ax.set_yscale("log"); ax.axhline(1, color="#52514e", lw=1, ls=(0, (4, 2)))
ax.set_xticks(x); ax.set_xticklabels(names); ax.set_ylabel("Normalised real-time residual")
ax.legend(frameon=False, ncol=3); plt.tight_layout(); plt.savefig("figs/fig4_consistency.png", dpi=220); plt.show()